# Atividade — Naive Bayes *from scratch* (10,0 pontos)

**Objetivo:** implementar um classificador Multinomial Naive Bayes para Spam/Não Spam usando apenas as ideias vistas na Aula 7.

**Regras**
- Não use `sklearn.naive_bayes` na parte obrigatória.
- Utilize `alpha = 1` na versão final.
- O código deve funcionar para qualquer mensagem de entrada.
- Responda às questões conceituais em células Markdown.
- Você pode reutilizar bibliotecas para manipulação de dados e tokenização; o **cálculo do Naive Bayes deve ser seu**.

## Critérios de avaliação

| Etapa | Pontos |
|---|---:|
| 1. Leitura dos dados e identificação correta das classes | 0,5 |
| 2. Pré-processamento coerente com a aula | 1,5 |
| 3. Vocabulário e tabela de frequências por classe | 1,5 |
| 4. Probabilidades a priori `P(classe)` | 1,0 |
| 5. Probabilidades condicionais com Laplace (`alpha=1`) | 2,0 |
| 6. Predição de uma nova mensagem + regra MAP | 2,0 |
| 7. Demonstração/explicação do problema de zero-frequência | 1,0 |
| 8. Interpretação do modelo e escolha da variante de NB | 0,5 |
| **Total** | **10,0** |

A pontuação privilegia os conceitos efetivamente desenvolvidos na aula: independência condicional, prior, verossimilhança, MAP, zero-frequência, Laplace e Multinomial NB.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import math
import re

import pandas as pd

# Compact Portuguese stop-word list to keep the notebook self-contained and offline.
STOP_WORDS = {
    "a", "ao", "aos", "aquela", "aquelas", "aquele", "aqueles", "aquilo",
    "as", "até", "com", "como", "da", "das", "de", "dela", "delas", "dele",
    "deles", "depois", "do", "dos", "e", "ela", "elas", "ele", "eles", "em",
    "entre", "era", "eram", "essa", "essas", "esse", "esses", "esta", "está",
    "estamos", "estão", "estas", "estava", "estavam", "este", "estes", "eu",
    "foi", "for", "foram", "fosse", "há", "isso", "isto", "já", "lhe", "lhes",
    "mais", "mas", "me", "mesmo", "meu", "meus", "minha", "minhas", "muito",
    "na", "nas", "não", "nem", "no", "nos", "nós", "nossa", "nossas", "nosso",
    "nossos", "num", "numa", "o", "os", "ou", "para", "pela", "pelas", "pelo",
    "pelos", "por", "porque", "qual", "quando", "que", "quem", "se", "sem",
    "seu", "seus", "sua", "suas", "só", "também", "te", "tem", "têm", "ter",
    "tu", "um", "uma", "você", "vocês", "vos"
}

TOKEN_PATTERN = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", re.UNICODE)

## 1. Leitura dos dados — 0,5 ponto

Implemente uma função que leia `email_aula7.txt` e retorne as mensagens separadas em duas classes: `Spam` e `NaoSpam`.

**Verificação esperada:** o arquivo possui 12 mensagens em cada classe.

In [8]:
def load_emails(filepath):
    emails = {"Spam": [], "NaoSpam": []}
    current = None
    with open(filepath, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("#"):
                if "(0)" in line:
                    current = "Spam"
                elif "(1)" in line:
                    current = "NaoSpam"
                continue
            if current is not None:
                emails[current].append(line)
    return emails


emails = load_emails("email_aula7.txt")
# Expected: {'Spam': 12, 'NaoSpam': 12}
print({label: len(msgs) for label, msgs in emails.items()})

{'Spam': 12, 'NaoSpam': 12}


## 2. Pré-processamento — 1,5 ponto

Implemente `preprocess(text)` para:
- converter para minúsculas;
- remover números;
- remover pontuação e caracteres especiais;
- remover stop words em português;
- retornar uma lista de tokens.

Explique em uma frase por que o mesmo pré-processamento deve ser aplicado no treinamento e na predição.

In [11]:
def preprocess(text):
    """Preprocess one e-mail and return a token list."""
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    tokens = TOKEN_PATTERN.findall(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return tokens


print(preprocess("Clique aqui e ganhe R$ 1000 agora!!!"))

['clique', 'aqui', 'ganhe', 'agora']


**Resposta:** TODO

## 3. Vocabulário e frequências — 1,5 ponto

Construa:
- o vocabulário `V`;
- a contagem de cada palavra em cada classe;
- o total de tokens em Spam e Não Spam.

Mostre uma tabela com pelo menos 10 palavras e suas contagens nas duas classes.

In [13]:
vocabulary = set()
word_count = {"Spam": Counter(), "NaoSpam": Counter()}
class_token_count = {"Spam": 0, "NaoSpam": 0}

for label, messages in emails.items():
    for msg in messages:
        tokens = preprocess(msg)
        class_token_count[label] += len(tokens)
        for tok in tokens:
            vocabulary.add(tok)
            word_count[label][tok] += 1

vocabulary = sorted(vocabulary)

print("|V| =", len(vocabulary))
print("Total tokens Spam    =", class_token_count["Spam"])
print("Total tokens NaoSpam =", class_token_count["NaoSpam"])

top_words = sorted(
    set(list(word_count["Spam"].keys()) + list(word_count["NaoSpam"].keys())),
    key=lambda w: word_count["Spam"][w] + word_count["NaoSpam"][w],
    reverse=True
)[:10]

df_freq = pd.DataFrame(
    {w: {"Spam": word_count["Spam"][w], "NaoSpam": word_count["NaoSpam"][w]} for w in top_words}
).T
df_freq

|V| = 112
Total tokens Spam    = 62
Total tokens NaoSpam = 64


,Spam,NaoSpam
aqui,2,1
ganhe,3,0
vivo,2,0
novo,2,0
favor,0,2
sistema,0,2
oferta,2,0
agora,2,0
reunião,0,2
todos,2,0


## 4. Probabilidades a priori — 1,0 ponto

Calcule:

$$
P(Spam) = \frac{N_{Spam}}{N}
\qquad
P(NaoSpam) = \frac{N_{NaoSpam}}{N}
$$

Mostre os dois valores e verifique se a soma é 1.

In [14]:
N = sum(len(msgs) for msgs in emails.values())
priors = {label: len(msgs) / N for label, msgs in emails.items()}
print(priors)
print("Soma =", sum(priors.values()))

{'Spam': 0.5, 'NaoSpam': 0.5}
Soma = 1.0


## 5. Probabilidades condicionais + Laplace — 2,0 pontos

Para cada palavra $w$ e classe $c$, implemente:

$$
P(w \mid c) = \frac{N_{w,c}+\alpha}{N_c+\alpha|V|}
$$

Use $\alpha = 1$.

Mostre $P(\text{"ganhe"} \mid Spam)$ e $P(\text{"ganhe"} \mid NaoSpam)$.

In [17]:
ALPHA = 1.0

def conditional_probability(word, label, alpha=ALPHA):
    vocab_size = len(vocabulary)
    numerator = word_count[label][word] + alpha
    denominator = class_token_count[label] + alpha * vocab_size
    return numerator / denominator

print("P('ganhe' | Spam)    =", conditional_probability("ganhe", "Spam"))
print("P('ganhe' | NaoSpam) =", conditional_probability("ganhe", "NaoSpam"))

P('ganhe' | Spam)    = 0.022988505747126436
P('ganhe' | NaoSpam) = 0.005681818181818182


## 6. Predição + MAP — 2,0 pontos

Implemente a classificação de uma mensagem nova usando:

$$
P(c \mid x) \propto P(c)\prod_w P(w \mid c)^{n(w,x)}
$$

Você pode implementar diretamente o produto ou, preferencialmente, usar logaritmos:

$$
score(c) = \log P(c) + \sum_w n(w,x)\log P(w \mid c)
$$

A classe prevista deve ser aquela com maior `score` (regra MAP).

**Atenção:** palavras repetidas devem contribuir de acordo com sua frequência.

In [27]:
def predict_scores(text, alpha=ALPHA):
    tokens = preprocess(text)
    counts = Counter(tokens)
    scores = {}
    for label in priors:
        score = math.log(priors[label])
        for word, n in counts.items():
            p = conditional_probability(word, label, alpha=alpha)
            score += n * math.log(p)
        scores[label] = score
    return scores

def predict(text, alpha=ALPHA):
    scores = predict_scores(text, alpha=alpha)
    return max(scores, key=scores.get)

mensagens = [
    "Parabéns, você ganhou um carro novo!",
    "Podemos remarcar a reunião para amanhã?",
    "Clique aqui e ganhe dinheiro dinheiro dinheiro!",
]

for m in mensagens:
    print(m)
    print(predict_scores(m))
    print("Classe:", predict(m))
    print()

Parabéns, você ganhou um carro novo!
{'Spam': -18.151314547070115, 'NaoSpam': -21.37508316071255}
Classe: Spam

Podemos remarcar a reunião para amanhã?
{'Spam': -21.329368377418064, 'NaoSpam': -18.197029330364607}
Classe: NaoSpam

Clique aqui e ganhe dinheiro dinheiro dinheiro!
{'Spam': -25.98451849571117, 'NaoSpam': -31.02290397022891}
Classe: Spam



## 7. Zero-frequência — 1,0 ponto

1. Execute sua implementação com $\alpha = 0$.
2. Procure uma mensagem que faça uma das classes receber probabilidade/score impossível por causa de uma palavra nunca observada naquela classe.
3. Explique por que isso acontece.
4. Execute novamente com $\alpha = 1$ e explique o que mudou.

Sua explicação deve usar explicitamente a expressão **problema de zero-frequência**.

In [28]:
def predict_scores(text, alpha=ALPHA):
    tokens = preprocess(text)
    counts = Counter(tokens)
    scores = {}
    for label in priors:
        score = math.log(priors[label])
        for word, n in counts.items():
            p = conditional_probability(word, label, alpha=alpha)
            if p <= 0:
                score = float("-inf")
                break
            score += n * math.log(p)
        scores[label] = score
    return scores


def predict(text, alpha=ALPHA):
    scores = predict_scores(text, alpha=alpha)
    return max(scores, key=scores.get)

In [29]:
## 7. Zero-frequência — 1,0 ponto

# Demonstração com `alpha = 0` vs `alpha = 1`

msg = "reunião projeto dinheiro"
print("alpha=0 ->", predict_scores(msg, alpha=0.0))
print("alpha=1 ->", predict_scores(msg, alpha=1.0))
print("Classe com alpha=1:", predict(msg, alpha=1.0))

alpha=0 -> {'Spam': -inf, 'NaoSpam': -inf}
alpha=1 -> {'Spam': -15.477165897643587, 'NaoSpam': -14.412839696446344}
Classe com alpha=1: NaoSpam


**Explicação:** TODO

## 8. Interpretação — 0,5 ponto

Responda:

**a)** Qual variante estudada na aula é a mais adequada para este problema: Multinomial, Bernoulli ou Gaussiano? Por quê?

**b)** Cite uma limitação do Naive Bayes discutida na aula.

**Resposta:** TODO

## Desafio opcional — não compõe os 10,0 pontos

Compare sua implementação com `sklearn.naive_bayes.MultinomialNB(alpha=1)`. Se houver divergência, investigue se ela está no pré-processamento, no vocabulário, nos *priors* ou na forma de representar as contagens.